# 06 导出 Power BI 数据表

本Notebook基于前面已经完成的数据清洗和分析结果，整理Power BI看板所需的数据表。

本节导出7张BI数据表：

1.`bi_trip_base_202407.csv`：骑行事件明细表，一行一次骑行，用于需求趋势、时间分布、用户类型、车型和骑行时长分析。

2.`bi_daily_demand_202407.csv`：日度需求聚合表，用于每日骑行趋势。

3.`bi_hourly_demand_202407.csv`：小时需求聚合表，用于工作日/周末和会员/临时用户小时曲线。

4.`bi_time_period_member_202407.csv`：用户类型与时段聚合表，用于对比会员和临时用户的使用时段结构。

5.`bi_bike_type_member_202407.csv`：用户类型与车型聚合表，用于分析车型偏好。

6.`bi_station_imbalance_202407.csv`：全日站点失衡表，一行一个站点，用于分析全日净流入、净流出、缺车风险和积车风险。

7.`bi_evening_station_imbalance_202407.csv`：晚高峰站点失衡表，一行一个站点，用于分析高峰期调度压力。

这种结构保留了明细分析的灵活性，同时把常用聚合指标和站点净流入/净流出指标直接提供给 Power BI 使用，避免在 BI 里重复写复杂计算逻辑。

## 1 导入依赖并设置路径

In [1]:
import pandas as pd
from pathlib import Path

DATA_CLEAN_DIR = Path("../data_clean")
OUTPUT_DIR = Path("../outputs")
BI_DIR = OUTPUT_DIR / "bi"

BI_DIR.mkdir(parents=True, exist_ok=True)

trip_base_path = DATA_CLEAN_DIR / "trip_base_202407.csv"
station_imbalance_path = DATA_CLEAN_DIR / "station_imbalance_202407.csv"
evening_station_imbalance_path = DATA_CLEAN_DIR / "evening_station_imbalance_202407.csv"

## 2 读取前面生成的数据表

本段读取02生成的骑行事件明细表，以及04生成的全日站点失衡表和晚高峰站点失衡表。

In [2]:
dtype_map = {
    "ride_id": "string",
    "rideable_type": "string",
    "start_station_name": "string",
    "start_station_id": "string",
    "end_station_name": "string",
    "end_station_id": "string",
    "member_casual": "string",
    "start_month": "string",
    "day_of_week": "string",
    "time_period": "string",
    "duration_group": "string",
}

trip_base = pd.read_csv(
    trip_base_path,
    dtype=dtype_map,
    parse_dates=["started_at", "ended_at", "start_date"],
)

station_imbalance = pd.read_csv(station_imbalance_path)
evening_station_imbalance = pd.read_csv(evening_station_imbalance_path)

print("trip_base记录数:", len(trip_base))
print("全日站点失衡表记录数:", len(station_imbalance))
print("晚高峰站点失衡表记录数:", len(evening_station_imbalance))

display(trip_base.head())
display(station_imbalance.head())
display(evening_station_imbalance.head())

trip_base记录数: 4720941
全日站点失衡表记录数: 2194
晚高峰站点失衡表记录数: 2178


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,...,time_period,duration_group,has_start_station,has_end_station,has_station_pair,has_start_location,has_end_location,has_location_pair,is_station_sample,is_map_sample
0,984F632114B98410,electric_bike,2024-07-11 13:32:52.359,2024-07-11 13:41:04.825,9 Ave & W 18 St,6190.08,9 Ave & W 33 St,6492.08,40.743174,-74.003664,...,daytime,5_10min,True,True,True,True,True,True,True,True
1,9B0E546FDB460C0E,electric_bike,2024-07-13 13:18:42.179,2024-07-13 13:22:46.631,W 42 St & 6 Ave,6517.08,W 49 St & 8 Ave,6747.06,40.754920,-73.984550,...,daytime,0_5min,True,True,True,True,True,True,True,True
2,6B939445A283D985,classic_bike,2024-07-08 20:34:27.848,2024-07-08 20:41:46.350,8 Ave & W 52 St,6816.07,9 Ave & W 33 St,6492.08,40.763707,-73.985162,...,night,5_10min,True,True,True,True,True,True,True,True
3,49444E058931E427,electric_bike,2024-07-14 15:42:44.695,2024-07-14 15:55:54.771,W 120 St & Claremont Ave,7745.07,W 78 St & Broadway,7311.07,40.810949,-73.963400,...,daytime,10_20min,True,True,True,True,True,True,True,True
4,74033CB639411DA0,classic_bike,2024-07-09 08:23:38.797,2024-07-09 08:28:48.647,W 42 St & 6 Ave,6517.08,W 49 St & 8 Ave,6747.06,40.754920,-73.984550,...,morning_peak,5_10min,True,True,True,True,True,True,True,True


,station_name,start_rides,station_lat,station_lng,end_rides,total_activity,net_inflow,abs_net_inflow,imbalance_ratio,imbalance_type
0,1 Ave & E 110 St,2728,40.792327,-73.938300,2744,5472,16,16,0.002924,net_inflow_accumulation
1,1 Ave & E 118 St,2040,40.797459,-73.934499,2053,4093,13,13,0.003176,net_inflow_accumulation
2,1 Ave & E 16 St,6963,40.732219,-73.981656,6697,13660,-266,266,-0.019473,net_outflow_shortage
3,1 Ave & E 18 St,7443,40.733812,-73.980544,7455,14898,12,12,0.000805,net_inflow_accumulation
4,1 Ave & E 30 St,5422,40.741444,-73.975361,5436,10858,14,14,0.001289,net_inflow_accumulation


,station_name,start_rides,station_lat,station_lng,end_rides,total_activity,net_inflow,abs_net_inflow,imbalance_ratio,imbalance_type
0,1 Ave & E 110 St,797,40.792327,-73.938300,880,1677,83,83,0.049493,net_inflow_accumulation
1,1 Ave & E 118 St,581,40.797459,-73.934499,725,1306,144,144,0.110260,net_inflow_accumulation
2,1 Ave & E 16 St,1923,40.732219,-73.981656,2258,4181,335,335,0.080124,net_inflow_accumulation
3,1 Ave & E 18 St,2380,40.733812,-73.980544,2964,5344,584,584,0.109281,net_inflow_accumulation
4,1 Ave & E 30 St,1363,40.741444,-73.975361,1086,2449,-277,277,-0.113107,net_outflow_shortage


**观察与总结**

本节成功读取三张基础数据表。`trip_base_202407.csv` 保持一行一次骑行事件的明细粒度，是 Power BI 看板中的主表。

`station_imbalance_202407.csv` 和 `evening_station_imbalance_202407.csv` 是站点级聚合表，已经包含 `start_rides`、`end_rides`、`net_inflow`、`abs_net_inflow` 和 `imbalance_type` 等核心字段，可直接用于站点失衡分析。

## 3 导出骑行事件明细表

本段导出 Power BI 主明细表。该表保持一行一次骑行事件的粒度，可支持日期、小时、星期、用户类型、车型、骑行时长和站点维度的交互式分析。

In [3]:
bi_trip_base = trip_base.copy()

bi_trip_base.to_csv(
    BI_DIR / "bi_trip_base_202407.csv",
    index=False,
)

print("已导出:", BI_DIR / "bi_trip_base_202407.csv")
print("记录数:", len(bi_trip_base))
display(bi_trip_base.head())

已导出: ..\outputs\bi\bi_trip_base_202407.csv
记录数: 4720941


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,...,time_period,duration_group,has_start_station,has_end_station,has_station_pair,has_start_location,has_end_location,has_location_pair,is_station_sample,is_map_sample
0,984F632114B98410,electric_bike,2024-07-11 13:32:52.359,2024-07-11 13:41:04.825,9 Ave & W 18 St,6190.08,9 Ave & W 33 St,6492.08,40.743174,-74.003664,...,daytime,5_10min,True,True,True,True,True,True,True,True
1,9B0E546FDB460C0E,electric_bike,2024-07-13 13:18:42.179,2024-07-13 13:22:46.631,W 42 St & 6 Ave,6517.08,W 49 St & 8 Ave,6747.06,40.754920,-73.984550,...,daytime,0_5min,True,True,True,True,True,True,True,True
2,6B939445A283D985,classic_bike,2024-07-08 20:34:27.848,2024-07-08 20:41:46.350,8 Ave & W 52 St,6816.07,9 Ave & W 33 St,6492.08,40.763707,-73.985162,...,night,5_10min,True,True,True,True,True,True,True,True
3,49444E058931E427,electric_bike,2024-07-14 15:42:44.695,2024-07-14 15:55:54.771,W 120 St & Claremont Ave,7745.07,W 78 St & Broadway,7311.07,40.810949,-73.963400,...,daytime,10_20min,True,True,True,True,True,True,True,True
4,74033CB639411DA0,classic_bike,2024-07-09 08:23:38.797,2024-07-09 08:28:48.647,W 42 St & 6 Ave,6517.08,W 49 St & 8 Ave,6747.06,40.754920,-73.984550,...,morning_peak,5_10min,True,True,True,True,True,True,True,True


## 4 导出站点失衡表

本段导出站点失衡分析表。站点失衡表保持一行一个站点的粒度，用于 Power BI 中展示站点净流入、净流出、缺车风险和积车风险。

In [4]:
station_imbalance.to_csv(
    BI_DIR / "bi_station_imbalance_202407.csv",
    index=False,
)

evening_station_imbalance.to_csv(
    BI_DIR / "bi_evening_station_imbalance_202407.csv",
    index=False,
)

print("已导出:", BI_DIR / "bi_station_imbalance_202407.csv")
print("已导出:", BI_DIR / "bi_evening_station_imbalance_202407.csv")

display(station_imbalance.head())
display(evening_station_imbalance.head())

已导出: ..\outputs\bi\bi_station_imbalance_202407.csv
已导出: ..\outputs\bi\bi_evening_station_imbalance_202407.csv


,station_name,start_rides,station_lat,station_lng,end_rides,total_activity,net_inflow,abs_net_inflow,imbalance_ratio,imbalance_type
0,1 Ave & E 110 St,2728,40.792327,-73.938300,2744,5472,16,16,0.002924,net_inflow_accumulation
1,1 Ave & E 118 St,2040,40.797459,-73.934499,2053,4093,13,13,0.003176,net_inflow_accumulation
2,1 Ave & E 16 St,6963,40.732219,-73.981656,6697,13660,-266,266,-0.019473,net_outflow_shortage
3,1 Ave & E 18 St,7443,40.733812,-73.980544,7455,14898,12,12,0.000805,net_inflow_accumulation
4,1 Ave & E 30 St,5422,40.741444,-73.975361,5436,10858,14,14,0.001289,net_inflow_accumulation


,station_name,start_rides,station_lat,station_lng,end_rides,total_activity,net_inflow,abs_net_inflow,imbalance_ratio,imbalance_type
0,1 Ave & E 110 St,797,40.792327,-73.938300,880,1677,83,83,0.049493,net_inflow_accumulation
1,1 Ave & E 118 St,581,40.797459,-73.934499,725,1306,144,144,0.110260,net_inflow_accumulation
2,1 Ave & E 16 St,1923,40.732219,-73.981656,2258,4181,335,335,0.080124,net_inflow_accumulation
3,1 Ave & E 18 St,2380,40.733812,-73.980544,2964,5344,584,584,0.109281,net_inflow_accumulation
4,1 Ave & E 30 St,1363,40.741444,-73.975361,1086,2449,-277,277,-0.113107,net_outflow_shortage


## 4.1 导出需求与用户结构聚合表


In [5]:
daily_demand = (
    trip_base
    .groupby(["start_date", "day_of_week", "day_of_week_num", "is_weekend"], as_index=False)
    .agg(
        rides=("ride_id", "count"),
        avg_duration_min=("ride_duration_min", "mean"),
        median_duration_min=("ride_duration_min", "median"),
    )
    .sort_values("start_date")
)

hourly_demand = (
    trip_base
    .groupby(["start_hour", "is_weekend", "member_casual"], as_index=False)
    .agg(
        rides=("ride_id", "count"),
        avg_duration_min=("ride_duration_min", "mean"),
    )
    .sort_values(["is_weekend", "member_casual", "start_hour"])
)

time_period_member = (
    trip_base
    .groupby(["time_period", "member_casual"], as_index=False)
    .agg(
        rides=("ride_id", "count"),
        avg_duration_min=("ride_duration_min", "mean"),
    )
)

bike_type_member = (
    trip_base
    .groupby(["rideable_type", "member_casual"], as_index=False)
    .agg(
        rides=("ride_id", "count"),
        avg_duration_min=("ride_duration_min", "mean"),
    )
)

daily_demand.to_csv(BI_DIR / "bi_daily_demand_202407.csv", index=False)
hourly_demand.to_csv(BI_DIR / "bi_hourly_demand_202407.csv", index=False)
time_period_member.to_csv(BI_DIR / "bi_time_period_member_202407.csv", index=False)
bike_type_member.to_csv(BI_DIR / "bi_bike_type_member_202407.csv", index=False)

print("Exported:", BI_DIR / "bi_daily_demand_202407.csv")
print("Exported:", BI_DIR / "bi_hourly_demand_202407.csv")
print("Exported:", BI_DIR / "bi_time_period_member_202407.csv")
print("Exported:", BI_DIR / "bi_bike_type_member_202407.csv")
display(daily_demand.head())
display(hourly_demand.head())


Exported: ..\outputs\bi\bi_daily_demand_202407.csv
Exported: ..\outputs\bi\bi_hourly_demand_202407.csv
Exported: ..\outputs\bi\bi_time_period_member_202407.csv
Exported: ..\outputs\bi\bi_bike_type_member_202407.csv


,start_date,day_of_week,day_of_week_num,is_weekend,rides,avg_duration_min,median_duration_min
0,2024-07-01,Monday,0,False,151980,14.036613,9.658075
1,2024-07-02,Tuesday,1,False,162119,13.717126,9.576317
2,2024-07-03,Wednesday,2,False,155842,13.906019,9.606475
3,2024-07-04,Thursday,3,False,117922,17.934757,11.164383
4,2024-07-05,Friday,4,False,111116,14.773023,9.673550


,start_hour,is_weekend,member_casual,rides,avg_duration_min
0,0,False,casual,14181,20.186407
4,1,False,casual,7953,20.846439
8,2,False,casual,4783,19.568869
12,3,False,casual,2968,20.231419
16,4,False,casual,2785,18.045828
